# Product Line Profitability & Margin Performance Analysis — Nassau Candy Distributor

This notebook performs the project EDA, profitability calculations, product/division analysis, margin-risk diagnostics, Pareto concentration analysis and cost diagnostics.

**Scope note:** the supplied project page contains a shipping-route conclusion, but the project title and methodology focus on product profitability. This notebook follows the profitability scope.


In [ ]:
!pip -q install pandas numpy matplotlib seaborn scipy plotly
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
from IPython.display import display
print("Libraries loaded.")


## 1. Upload and load the CSV

In [ ]:
uploaded = files.upload()
file_name = next(iter(uploaded))
df = pd.read_csv(file_name, encoding="latin1")
print("Dataset shape:", df.shape)
display(df.head())


## 2. Data validation

In [ ]:
print("Missing cells:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print("Non-positive sales:", int((df["Sales"] <= 0).sum()))
print("Non-positive units:", int((df["Units"] <= 0).sum()))
print("Gross profit mismatches:", int((~np.isclose(df["Gross Profit"], df["Sales"] - df["Cost"])).sum()))


## 3. Profitability metrics

In [ ]:
df["Order Date"] = pd.to_datetime(df["Order Date"], dayfirst=True, errors="coerce")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], dayfirst=True, errors="coerce")
df["Gross_Margin_Pct"] = df["Gross Profit"] / df["Sales"] * 100
df["Profit_per_Unit"] = df["Gross Profit"] / df["Units"]

product = (
    df.groupby(["Product ID","Product Name"], as_index=False)
      .agg(Division=("Division","first"), Sales=("Sales","sum"), Units=("Units","sum"),
           Gross_Profit=("Gross Profit","sum"), Cost=("Cost","sum"),
           Orders=("Order ID","nunique"))
)
product["Gross_Margin_Pct"] = product["Gross_Profit"]/product["Sales"]*100
product["Profit_per_Unit"] = product["Gross_Profit"]/product["Units"]

total_sales = product["Sales"].sum()
total_profit = product["Gross_Profit"].sum()
product["Revenue_Contribution_Pct"] = product["Sales"]/total_sales*100
product["Profit_Contribution_Pct"] = product["Gross_Profit"]/total_profit*100

print(f"Total sales: {total_sales:,.2f}")
print(f"Total gross profit: {total_profit:,.2f}")
print(f"Overall gross margin: {total_profit/total_sales*100:.2f}%")


## 4. Product profitability

In [ ]:
display(
    product.sort_values("Gross_Profit", ascending=False)[
        ["Product Name","Division","Sales","Gross_Profit","Gross_Margin_Pct",
         "Profit_per_Unit","Revenue_Contribution_Pct","Profit_Contribution_Pct"]
    ].round(3)
)


## 5. Division performance

In [ ]:
division = (
    df.groupby("Division", as_index=False)
      .agg(Sales=("Sales","sum"), Gross_Profit=("Gross Profit","sum"), Cost=("Cost","sum"),
           Units=("Units","sum"), Orders=("Order ID","nunique"))
)
division["Gross_Margin_Pct"] = division["Gross_Profit"]/division["Sales"]*100
division["Profit_per_Unit"] = division["Gross_Profit"]/division["Units"]
division["Revenue_Contribution_Pct"] = division["Sales"]/total_sales*100
division["Profit_Contribution_Pct"] = division["Gross_Profit"]/total_profit*100
display(division.round(3))


## 6. High-sales / low-margin diagnostic

In [ ]:
sales_median = product["Sales"].median()
margin_median = product["Gross_Margin_Pct"].median()

high_sales_low_margin = product[
    (product["Sales"] >= sales_median) &
    (product["Gross_Margin_Pct"] < margin_median)
].sort_values("Sales", ascending=False)

print("Median product sales:", round(sales_median, 2))
print("Median product gross margin:", round(margin_median, 2), "%")
display(high_sales_low_margin[
    ["Product Name","Sales","Gross_Profit","Gross_Margin_Pct","Profit_per_Unit"]
].round(3))


## 7. Pareto concentration

In [ ]:
pareto_profit = product.sort_values("Gross_Profit", ascending=False).reset_index(drop=True)
pareto_profit["Cum_Profit_Pct"] = pareto_profit["Profit_Contribution_Pct"].cumsum()
pareto_revenue = product.sort_values("Sales", ascending=False).reset_index(drop=True)
pareto_revenue["Cum_Revenue_Pct"] = pareto_revenue["Revenue_Contribution_Pct"].cumsum()

n80_profit = int((pareto_profit["Cum_Profit_Pct"] < 80).sum() + 1)
n80_revenue = int((pareto_revenue["Cum_Revenue_Pct"] < 80).sum() + 1)

print(f"Top 5 profit contribution: {pareto_profit.head(5)['Profit_Contribution_Pct'].sum():.2f}%")
print(f"Top 5 revenue contribution: {pareto_revenue.head(5)['Revenue_Contribution_Pct'].sum():.2f}%")
print(f"Products needed for 80% profit: {n80_profit}/{len(product)}")
print(f"Products needed for 80% revenue: {n80_revenue}/{len(product)}")


## 8. Margin volatility

In [ ]:
df["YearMonth"] = df["Order Date"].dt.to_period("M").astype(str)
monthly = (
    df.groupby(["Product ID","Product Name","YearMonth"], as_index=False)
      .agg(Sales=("Sales","sum"), Gross_Profit=("Gross Profit","sum"))
)
monthly["Margin_Pct"] = monthly["Gross_Profit"]/monthly["Sales"]*100
volatility = (
    monthly.groupby(["Product ID","Product Name"], as_index=False)["Margin_Pct"]
           .std(ddof=0)
           .rename(columns={"Margin_Pct":"Margin_Volatility_Ppt"})
)
display(volatility.fillna(0).round(4))


## 9. Cost vs sales

In [ ]:
corr = product[["Cost","Sales"]].corr().iloc[0,1]
print(f"Product-level Cost vs Sales Pearson correlation: {corr:.4f}")

plt.figure(figsize=(9,6))
plt.scatter(product["Cost"], product["Sales"])
plt.xlabel("Cost")
plt.ylabel("Sales")
plt.title("Product Cost vs Sales")
plt.show()


## 10. Interpretation checklist

- Use gross margin and absolute gross profit together.
- Treat high-sales/low-margin products as review signals rather than automatic discontinuation decisions.
- Interpret Pareto concentration as portfolio dependency.
- Zero margin volatility means the observed product margin did not vary across the available months.
- Correlation does not establish causation.
